# PlanAdherenceMetric

## What it measures

Whether the agent's execution followed the plan it declared. DeepEval extracts a task and a
plan from the agent's **trace**, then judges whether the trace's actual steps match that
plan - both that planned steps happened, and that unplanned steps did not.

It is the only metric in this suite that reads a trace rather than a request/response pair,
and the only one that depends on DeepEval-native instrumentation being switched on inside
the application.

## When it is useful

On agents that plan before acting, where the interesting failure is drift: the plan says
screen both parties, execution screens one; or the agent improvises a step nobody
authorised. It is a governance metric as much as a quality one - in a regulated workflow,
"the system did something it did not declare" is a finding on its own.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `LLMTestCase` |
| `input` | yes |
| `actual_output` | yes |
| `_trace_dict` | yes in practice - the plan and the executed steps are both read from here |

`_trace_dict` is a **private attribute** on `LLMTestCase`. DeepEval populates it
automatically when a metric runs inside its own `@observe` tracing context. A detached
black-box harness has no such context, so this notebook assigns it explicitly from trace
data the API serves. That coupling to a private attribute is a real portability risk and is
recorded in `metric-notes.md`.

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
EXPECTED_SCHEMA_VERSION = "1.0.0"

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    if served and served != EXPECTED_SCHEMA_VERSION:
        print(f"WARNING: application reports contract version {served}, these "
              f"notebooks were written against {EXPECTED_SCHEMA_VERSION}. "
              f"Field names may have changed - see docs/evaluation-contract.md.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoints exercised

| Endpoint | Role here |
|---|---|
| `POST /api/cases/{case_id}/investigate` | Runs the agent |
| `GET /api/agent/trace/{run_id}` | Serves `deepeval_trace` (the span tree) plus `tools_available`, `tools_selected`, `skipped_tools` and the ordered `steps[]` timeline |
| `GET /api/eval/export/{run_id}` | Supplies `input` and `actual_output` |

### A prerequisite that is easy to miss

`deepeval_trace` is `null` unless the application was started with its optional `eval`
extra installed **and** `EVAL_TRACING=true`. The application's shipped container image does
not install that extra, so a default Docker deployment returns `null` here and this
notebook cannot run. `GET /api/health` reports `eval_tracing`, and the cell below checks it
before anything else, so the failure is named rather than mysterious.

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

In [ ]:
# --------------------------------------------------------------------------
# Prerequisite check: is DeepEval-native tracing switched on?
# --------------------------------------------------------------------------
health = api("GET", "/api/health")
print(f"eval_tracing reported by the application: {health.get('eval_tracing')}")

if not health.get("eval_tracing"):
    raise RuntimeError(
        "The application reports eval_tracing: false, so GET /api/agent/trace/{run_id} "
        "will return deepeval_trace: null and this metric has nothing to read.\n\n"
        "To enable it, the application must be started with its optional eval extra and "
        "tracing on:\n"
        "    uv sync --extra eval\n"
        "    EVAL_TRACING=true <start the backend>\n\n"
        "The shipped container image builds with `uv sync --frozen --no-dev` and omits the "
        "extra, so a default Docker deployment will always report false. See "
        "metric-notes.md -> 'Required application change'.\n"
        "Note also that tracing is ignored entirely when ENVIRONMENT=production."
    )

In [ ]:
# --------------------------------------------------------------------------
# The exact request.
#
# This endpoint reads no request body: the case_id in the path is the entire
# input and all context is loaded server-side.
# --------------------------------------------------------------------------
CASE_ID = SCENARIOS["s3"]["case_id"]   # "Possible sanctions name match (beneficiary near-miss)"

print("POST", f"{API_BASE}/api/cases/{CASE_ID}/investigate")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
print("body: (none)")

In [ ]:
# --------------------------------------------------------------------------
# The raw responses.
# --------------------------------------------------------------------------
investigation = api("POST", f"/api/cases/{CASE_ID}/investigate", expect_status=201)
RUN_ID = investigation["run_id"]
print(f"run_id = {RUN_ID}\n")

trace = api("GET", f"/api/agent/trace/{RUN_ID}", role="eval_reader")
export = api("GET", f"/api/eval/export/{RUN_ID}", role="eval_reader")

print(f"run status : {trace['status']}   error: {trace['error']}")
print(f"model      : {trace['model']}   prompt_version: {trace['prompt_version']}")
print()

deepeval_trace = trace.get("deepeval_trace")
if not deepeval_trace:
    raise RuntimeError(
        f"GET /api/agent/trace/{RUN_ID} returned deepeval_trace: null even though "
        f"/api/health reported eval_tracing: true. The run may predate tracing being "
        f"enabled - re-run this notebook so the investigation happens under tracing."
    )

print("deepeval_trace top-level keys:", sorted(deepeval_trace.keys()))
print()


def walk(span, depth=0):
    pad = "  " * depth
    print(f"{pad}{span.get('span_type', 'BaseSpan'):<14} {span.get('name')}")
    print(f"{pad}    input : {json.dumps(span.get('input'))[:110]}")
    print(f"{pad}    output: {json.dumps(span.get('output'))[:110]}")
    for child in span.get("children") or []:
        walk(child, depth + 1)


print("span tree:")
for root in deepeval_trace.get("root_spans") or []:
    walk(root, 1)

## Mapping the API response onto DeepEval fields

| DeepEval field | Source |
|---|---|
| `input` | `input` from the eval export |
| `actual_output` | `actual_output` from the eval export |
| `_trace_dict` | `deepeval_trace`, plus the plan fields from the trace endpoint |

### Why the span tree alone is not enough

`deepeval_trace` records what *happened* - retriever, tool and LLM spans - but carries no
planning span, because the application's planner is a rule engine rather than an LLM and
emits no reasoning artefact. Handed only the span tree, the metric asks a judge to infer a
plan from the very trace it will then score against that plan. That is circular, and in
authoring runs it produced a plausible-looking `0.5` derived from nothing the application
ever declared.

The application does publish a real, ex-ante plan - just not inside the span tree:

- `tools_selected` - the tool calls the planner chose **before** any tool ran, each with a
  stated `reason`
- `skipped_tools` - the tools it deliberately did not choose, each with a reason
- `retrieval_runs` and the `steps[]` timeline, whose `type` enum (`retrieval`, `tool_call`,
  `synthesis`) documents the three unconditional stages of the workflow

The cell below composes those API fields into a `declared_plan` and attaches it alongside
the span tree. Every element comes from the API; nothing is invented. The one editorial act
is including the retrieval and synthesis stages, which the planner does not choose because
they are unconditional - the `steps[]` type enum is the published evidence for that.

In [ ]:
# --------------------------------------------------------------------------
# Composing the declared plan from API fields.
#
# Sources, in order:
#   retrieval_runs[].query   -> the retrieval stage actually issued
#   tools_selected[]         -> the planner's ex-ante tool choices, with reasons
#   skipped_tools[]          -> tools deliberately not chosen, with reasons
#   steps[] type enum        -> evidence that synthesis is an unconditional stage
# --------------------------------------------------------------------------
declared_plan = []

for run in trace["retrieval_runs"]:
    declared_plan.append(
        f"Retrieve supporting context for: {run['query']}")

for selected in trace["tools_selected"]:
    declared_plan.append(
        f"Call {selected['server']}.{selected['tool']} with "
        f"{json.dumps(selected['arguments'])} because {selected['reason']}")

for skipped in trace["skipped_tools"] or []:
    declared_plan.append(
        f"Deliberately skip {skipped['server']}.{skipped['tool']} because "
        f"{skipped['reason']}")

declared_plan.append(
    "Synthesize a risk assessment and a recommended action from the retrieved context "
    "and the tool results.")

print("EXPECTED PLAN (declared by the application, assembled from API fields)")
for n, step in enumerate(declared_plan, start=1):
    print(f"  {n}. {textwrap.fill(step, width=92, subsequent_indent='     ')}")

print()
print("ACTUAL PLAN / executed timeline (GET /api/agent/trace -> steps[])")
for step in trace["steps"]:
    print(f"  {step['sequence']:>2}. {step['type']:<10} status={step['status']:<10} "
          f"latency_ms={step['latency_ms']}  detail={json.dumps(step['detail'])[:90]}")

if not trace["tools_selected"]:
    raise RuntimeError(
        "The trace reports no tools_selected, so there is no declared plan to judge "
        "adherence against. On a repeat run for the same case the planner skips tools it "
        "already completed - set AML_RESET_BEFORE_RUN=true and re-run."
    )

In [ ]:
# --------------------------------------------------------------------------
# Build the test case and attach the trace.
#
# _trace_dict is a DeepEval private attribute. It is assigned directly because
# a detached harness never runs inside DeepEval's own @observe context, which
# is what would normally populate it. See the limitations section.
# --------------------------------------------------------------------------
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=export["input"],
    actual_output=export["actual_output"],
)
test_case._trace_dict = {**deepeval_trace, "declared_plan": declared_plan}

print("USER INPUT")
print(" ", test_case.input)
print()
print("ACTUAL OUTPUT")
print(textwrap.fill(test_case.actual_output, width=96, initial_indent="  ",
                    subsequent_indent="  ")[:900])
print()
print(f"TRACE ATTACHED: {len(json.dumps(test_case._trace_dict))} bytes, "
      f"keys={sorted(test_case._trace_dict.keys())}")
print(f"  declared_plan steps : {len(declared_plan)}")
print(f"  span tree roots     : {len(test_case._trace_dict.get('root_spans') or [])}")

## Judge and threshold

- **Judge model**: `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `PlanAdherenceMetric`.

The default is kept, and here it is doing real work rather than being a placeholder. The
metric penalises execution that goes *beyond* the plan as well as execution that falls
short, and a trace of a real agent always contains framing the plan does not enumerate. In
authoring runs a fully compliant execution scored `0.5` for that reason. Setting the
threshold higher would fail correct behaviour; the useful signal is a drop *below* this
level across runs, not the absolute number.

`verbose_mode=True` matters more than usual on this metric: the verbose log prints the plan
DeepEval actually extracted, which is the only way to tell "the agent drifted" from "the
plan was not read as intended".

In [ ]:
from deepeval.metrics import PlanAdherenceMetric

metric = PlanAdherenceMetric(
    threshold=0.5,          # DeepEval's documented default
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
    verbose_mode=True,      # prints the extracted plan - essential for this metric
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Guard against a vacuous pass.
#
# When PlanAdherenceMetric can extract no plan it returns score = 1 with the
# reason "There were no plans to evaluate...". That is a non-answer, and it
# looks exactly like a perfect pass on a results table. Turn it into a failure.
# --------------------------------------------------------------------------
if metric.reason and "no plans to evaluate" in metric.reason.lower():
    raise RuntimeError(
        "PlanAdherenceMetric extracted no plan from the trace and returned score = 1 "
        "as a default. That is a vacuous pass, not a result.\n"
        f"reason: {metric.reason}\n"
        "Check that declared_plan was attached to _trace_dict and that tools_selected "
        "was non-empty."
    )
print("Plan extraction produced a non-empty plan - the score below is a real result.")

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

## Limitations in a black-box acceptance test

1. **The application's plan is rule-based, not model-authored.** The planner is a
   deterministic rule engine, so this metric is measuring a rule engine's consistency with
   itself rather than an LLM's ability to follow its own reasoning - the failure mode the
   metric was designed for. The application's own evaluation contract says as much. It
   still catches real execution drift (a selected tool that never ran, an unplanned call),
   which is worth having; it just is not the agentic-reasoning signal the name suggests.
2. **It depends on a DeepEval private attribute.** `_trace_dict` is not public API. A
   DeepEval upgrade that renames or repurposes it breaks this notebook silently - most
   likely into the vacuous-pass mode the guard cell above exists to catch.
3. **It depends on optional application instrumentation.** No `eval` extra or
   `EVAL_TRACING=false` means no span tree, and the metric cannot run at all. Tracing is
   also ignored entirely when `ENVIRONMENT=production`, so this metric is structurally
   unavailable against a production deployment.
4. **Unconditional stages depress the score.** The judge treats retrieval and synthesis as
   steps beyond the plan even when they are enumerated, because they are workflow
   scaffolding rather than planned actions. A correct run therefore does not score `1.0`,
   and the absolute number is not comparable to another system's.
5. **The plan is assembled by the harness, not served as one object.** The application
   exposes the planner's decision across `tools_selected`, `skipped_tools`,
   `retrieval_runs` and `steps[]`, but publishes no single "plan" field. This notebook's
   composition is a reasonable reading of those fields, not a contract - a different
   harness could compose it differently and get a different score. Publishing an explicit
   plan object, or emitting a planning span, would remove that ambiguity; it is recorded as
   a required application change in `metric-notes.md`.